# Anchor Refinery

### Procedure
1. Select Julia Anchors within SGA 2025
2. Filter anchors to have diameter less than 40"
3. Select morphology to review
4. Save set of good anchors and bad anchors separately
5. Record anchor salvages (bad anchors made good)

##### (Edit: move to new notebook entirely)
6. In Generating_LG_VI_Set, add morphology confirmation mode via SGA 2025 Morphology Version 1
7. Review Bin 1 galaxies and add to good anchor set until 1100 reached

### 1.
Julia's anchors available in SGA 2025 are generated using Select_Galaxies.ipynb

In [1]:
# for debugging cutout_vetter
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd

import glob
import h5py
import os

from astropy.coordinates import SkyCoord # galaxy matching
from astropy.table import vstack
import astropy.units as u

import matplotlib.pyplot as plt
import matplotlib.image as mpimg # displaying images

from SGA.SGA import read_sga_sample # loading SGA-2025
from SGA.qa import sdss_rgb # displaying SGA-2025 images

from IPython.display import display, clear_output # for display functions
import ipywidgets as widgets # for buttons

from cutout_vetter import CutoutVetter # interactive image grid

In [3]:
HEADNUM = 30
MAX_SEP = 9.5
N_COLS = 10
N_PAIRS_PER_PAGE = 50
SSL_DIR = '/global/cfs/cdirs/desicollab/users/ioannis/SGA/2025/ssl'

In [4]:
def generate_SGA_2025(cutout_index):
    _, south = read_sga_sample(region="dr11-south", no_groups=True)
    _, north = read_sga_sample(region="dr11-north", no_groups=True)

    south["VI_REGION"] = "dr11-south"
    north["VI_REGION"] = "dr11-north"

    catalog = vstack([south, north]).to_pandas()
    catalog["cutout_key"] = list(zip(catalog["VI_REGION"], catalog["SGAID"].astype(int)))
    catalog = catalog[catalog["cutout_key"].isin(cutout_index)].copy()
    catalog.drop(columns="cutout_key", inplace=True)
    catalog.reset_index(drop=True, inplace=True)
    
    print("Total catalog:", len(catalog))
    print("South:", len(south))
    print("North:", len(north))

    return catalog

def SGA_2025_Anchors(anchors, SGA_2025, max_sep=MAX_SEP):

    anchor_coords = SkyCoord(
        ra=anchors["ra"].to_numpy(dtype=float),
        dec=anchors["dec"].to_numpy(dtype=float),
        unit="deg"
    )

    sample_coords = SkyCoord(
        ra=SGA_2025["RA"].to_numpy(dtype=float),
        dec=SGA_2025["DEC"].to_numpy(dtype=float),
        unit="deg"
    )

    # For every anchor, find the nearest SGA-2025 galaxy
    idx, sep2d, _ = anchor_coords.match_to_catalog_sky(sample_coords)

    max_sep_u = max_sep * u.arcsec
    good = sep2d < max_sep_u

    # SGA-2025 counterparts of successfully matched anchors
    filtered_anchors = SGA_2025.iloc[idx[good]].copy()

    # Add the anchor catalog's main_type
    filtered_anchors["Main_type"] = (anchors.iloc[np.where(good)[0]]["Main_type"].to_numpy())
    filtered_anchors["ref_id"] = anchors.iloc[np.where(good)[0]]["ref_id"].to_numpy()

    print(f"Matched {good.sum()} / {len(anchors)} anchors")
    print("Minimum:", sep2d.min().arcsec, "arcsec")
    print("Median :", np.median(sep2d.arcsec), "arcsec")
    print("Maximum:", sep2d.max().arcsec, "arcsec")

    return filtered_anchors, idx, sep2d

def getInfoMaxSep(anchors, SGA_2025, idx, sep2d, max_sep=MAX_SEP, headNum=HEADNUM, List=False):
    # Create a dataframe of anchor → SGA-2025 matches
    matched_results = pd.DataFrame({
        'anchor_idx': np.arange(len(anchors)),
        'sga_idx': idx,
        'separation_arcsec': sep2d.to(u.arcsec).value
    })

    # Keep only matches under the threshold
    valid_matches = matched_results[matched_results['separation_arcsec'] < max_sep]

    # Sort by separation in descending order
    furthest_matches = valid_matches.sort_values(by='separation_arcsec', ascending=False)

    # Grab the furthest successful matches
    top_seps = furthest_matches.head(headNum)

    # Get the corresponding SGA-2025 galaxies
    top_seps_SGA_2025 = SGA_2025.iloc[top_seps['sga_idx'].astype(int).tolist()].copy()
    top_seps_SGA_2025['sep_arcsec'] = (top_seps['separation_arcsec'].values)

    # Get the corresponding anchors
    top_seps_anchors = anchors.iloc[top_seps['anchor_idx'].astype(int).tolist()].copy()
    top_seps_anchors['sep_arcsec'] = (top_seps['separation_arcsec'].values)

    # Print the matches if requested
    for _, row in top_seps.iterrows():
        a_idx = int(row['anchor_idx'])
        s_idx = int(row['sga_idx'])

        if List:
            print(f"--- Separation: {row['separation_arcsec']:.3f} arcseconds ---")
            print("Anchor:", anchors.iloc[a_idx][['ra', 'dec']].to_dict())
            print("SGA-2025:", SGA_2025.iloc[s_idx][['RA', 'DEC']].to_dict())

    return top_seps_SGA_2025, top_seps_anchors

In [5]:
# Taken from /global/homes/q/qshimp/SGA/doc/tutorials/SGA-ssl.ipynb
def build_cutout_index(ssl_dir):
    """
    Return {(region, sgaid): (hdf5_path, row_index)}
    for fast image retrieval.
    """
    files = sorted(glob.glob(os.path.join(ssl_dir, "ssl-cutouts-dr11-*.hdf5")))
    if not files:
        raise FileNotFoundError(f"No cutout files found in {ssl_dir}")

    index = {}
    for f in files:
        filename = os.path.basename(f)
        
        if "dr11-south" in filename:
            region = "dr11-south"
        elif "dr11-north" in filename:
            region = "dr11-north"
        else:
            raise ValueError(f"Could not determine region from filename: {filename}")
            
        with h5py.File(f, "r") as H:
            sgaids = H["sgaid"][:]
            print(f"  {filename}: {len(sgaids):,} galaxies ({region})")
            
            for i, sgaid in enumerate(sgaids):
                key = (region, int(sgaid))
                if key in index:
                    print(f"WARNING: duplicate key found: {key}")
                index[key] = (f, i)

    print(f"Total unique (region, SGAID) pairs indexed: {len(index):,}")

    return index
 
# Taken from /global/homes/q/qshimp/SGA/doc/tutorials/SGA-ssl.ipynb
def show_cutout_grid(sgaids, cutout_index, ncols=4, figsize_per=2,
                     titles=None, data=None, title_fontsize=None):
    """
    Display a grid of galaxy cutouts given a list of SGAIDs.
    
    If `data` is provided and `titles` is None, each panel is labelled
    with (RA, Dec) from the catalog for easy viewer cross-referencing.
    `title_fontsize` defaults to max(6, int(figsize_per * 4)), scaling
    with cell size so titles remain readable regardless of grid density.
    
    """
    sgaids = [int(s) for s in sgaids]
    radec = {}
    
    if title_fontsize is None:
        title_fontsize = max(6, int(figsize_per * 4))
    
    if data is not None and titles is None:
        
        for s, ra, dec in zip(data['SGAID'], data['RA'], data['DEC']):
            radec[int(s)] = (float(ra), float(dec))

    nrows = int(np.ceil(len(sgaids) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(ncols * figsize_per, nrows * figsize_per)
    )
    axes = np.array(axes).ravel()

    for i, (ax, sgaid) in enumerate(zip(axes, sgaids)):
        
        region = None
        
        if data is not None:
            matches = data[data["SGAID"].astype(int) == sgaid]
            
            if len(matches) > 0:
                region = str(matches.iloc[0]["VI_REGION"])

        key = (region, sgaid) if region is not None else None
        
        if key is not None and key in cutout_index:
            fname, idx = cutout_index[key]
            
            with h5py.File(fname, "r") as H:
                img = H["images"][idx]
                
            rgb = sdss_rgb([img[0], img[1], img[2]], ["g", "r", "z"])
            ax.imshow(rgb, origin="lower")
            
        else:
            ax.text(0.5, 0.5, f'SGAID\n{sgaid}\nnot found',
                    ha='center', va='center', transform=ax.transAxes, fontsize=7)

        if titles is not None:
            title = str(titles[i])
            
        elif sgaid in radec:
            ra, dec = radec[sgaid]
            title = f'{sgaid} ({ra:.4f}, {dec:.4f})'
            
        else:
            title = str(sgaid)
            
        ax.set_title(title, fontsize=title_fontsize)
        ax.axis('off')

    for ax in axes[len(sgaids):]:
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()

# Load legacy survey jpgs
def load_jpg(row):
    tid = row["ref_id"]
    patternM = f"/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor_jpgs/Model/{tid}_*.jpg"
    matchesM = glob.glob(patternM)
    patternR = f"/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor_jpgs/Residual/{tid}_*.jpg"
    matchesR = glob.glob(patternR)
    patternI = f"/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor_jpgs/Image/{tid}_*.jpg"
    matchesI = glob.glob(patternI)

    model = np.flipud(mpimg.imread(matchesM[0]))
    residual = np.flipud(mpimg.imread(matchesR[0]))
    image = np.flipud(mpimg.imread(matchesI[0]))

    return model, residual, image

# Function to display galaxies
def display_galaxy(df, change=None):

    # Keep index in bounds
    i = index_box.value
    if i < 0 or i >= len(df):
        return

    # Get dataframe row
    row = df.iloc[i]

    # Load images
    path = row["Path"]
    model, residual, image = load_jpg(row)
    
    with image_out:
        image_out.clear_output(wait=True)
        view = view_selector.value

        if view == "Image":
            fig, ax = plt.subplots(figsize=(6,6))
            ax.imshow(image)
            ax.set_title("Image")

        elif view == "Model":
            fig, ax = plt.subplots(figsize=(6,6))
            ax.imshow(model)
            ax.set_title("Model")
            
        elif view == "Residuals":
            fig, ax = plt.subplots(figsize=(6,6))
            ax.imshow(residual)
            ax.set_title("Residuals")

        plt.show()

In [6]:
# Function to display group of galaxies
def display_match_group(sga_df, anchor_df, cutout_index, ncols=5, n_pairs_per_page=25, figsize_per=2.0, start=0):
    # Keep indices in bounds
    n_pairs = min(len(sga_df), len(anchor_df))

    if n_pairs == 0:
        print("No matched galaxies to display.")
        return

    start = max(0, min(start, n_pairs - 1))
    end = min(start + n_pairs_per_page, n_pairs)

    n_page = end - start
    n_groups = int(np.ceil(n_page / ncols))
    nrows = n_groups * 2
    
    sga_subset = sga_df.iloc[start:end]
    anchor_subset = anchor_df.iloc[start:end]

    n = len(sga_subset)

    # Create figure
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * figsize_per, nrows * figsize_per))
    axes = np.asarray(axes).reshape(nrows, ncols)

    for group in range(n_groups):

        group_start = start + group * ncols
        group_end = min(group_start + ncols, end)

        for col, pair_idx in enumerate(range(group_start, group_end)):
            sga_row = sga_df.iloc[pair_idx]
            anchor_row = anchor_df.iloc[pair_idx]
            sga_ax = axes[group * 2, col]
            anchor_ax = axes[group * 2 + 1, col]
            region = sga_row["VI_REGION"]
            sgaid = int(sga_row["SGAID"])   
            key = (region, sgaid)
            
            if key not in cutout_index:
                raise FileNotFoundError(f"{region} SGAID {sgaid} is missing from cutout_index.")
            
            fname, hdf_idx = cutout_index[key]

            with h5py.File(fname, "r") as H:
                img = H["images"][hdf_idx]

            rgb = sdss_rgb([img[0], img[1], img[2]], ["g", "r", "z"])
            sga_ax.imshow(rgb, origin="lower")
            sga_ax.set_title(f"{pair_idx + 1}: SGA {sgaid}", fontsize=8)
            sga_ax.axis("off")

            # ANCHOR IMAGE
            model, residual, image = load_jpg(anchor_row)

            anchor_ax.imshow(image)
            anchor_ax.set_title(f"{pair_idx + 1}: Anchor {anchor_row['ref_id']}", fontsize=8)
            anchor_ax.axis("off")

        # Hide unused columns
        for col in range(group_end - group_start, ncols):
            axes[group * 2, col].axis("off")
            axes[group * 2 + 1, col].axis("off")

    # Labels on left side
    for group in range(n_groups):
        axes[group * 2, 0].set_ylabel("SGA", fontsize=11, rotation=90)
        axes[group * 2 + 1, 0].set_ylabel("Anchor", fontsize=11, rotation=90)
        
    # Overall title
    fig.suptitle(f"Matches {start + 1}–{end} of {n_pairs}", fontsize=14)
    plt.tight_layout()
    plt.show()

def update_match_view(change=None):
    global current_start

    with viewer_output:
        clear_output(wait=True)

        display_match_group(top_seps_SGA_2025, top_seps_anchors, cutout_index, 
                            start=current_start, ncols=N_COLS, n_pairs_per_page=N_PAIRS_PER_PAGE)

def next_page(_):
    global current_start
    n_pairs = min(len(top_seps_SGA_2025), len(top_seps_anchors))    
    current_start = min(current_start + N_PAIRS_PER_PAGE, max(0, n_pairs - N_PAIRS_PER_PAGE))
    update_match_view()

def previous_page(_):
    global current_start
    current_start = max(0, current_start - N_PAIRS_PER_PAGE)
    update_match_view()

In [7]:
cutout_index = build_cutout_index(SSL_DIR)
SGA_2025 = generate_SGA_2025(cutout_index)

  ssl-cutouts-dr11-north-chunk0000.hdf5: 41,043 galaxies (dr11-north)
  ssl-cutouts-dr11-north-chunk0001.hdf5: 41,043 galaxies (dr11-north)
  ssl-cutouts-dr11-south-chunk0000.hdf5: 45,451 galaxies (dr11-south)
  ssl-cutouts-dr11-south-chunk0001.hdf5: 45,451 galaxies (dr11-south)
  ssl-cutouts-dr11-south-chunk0002.hdf5: 45,451 galaxies (dr11-south)
  ssl-cutouts-dr11-south-chunk0003.hdf5: 45,451 galaxies (dr11-south)
  ssl-cutouts-dr11-south-chunk0004.hdf5: 45,451 galaxies (dr11-south)
  ssl-cutouts-dr11-south-chunk0005.hdf5: 45,451 galaxies (dr11-south)
  ssl-cutouts-dr11-south-chunk0006.hdf5: 45,451 galaxies (dr11-south)
  ssl-cutouts-dr11-south-chunk0007.hdf5: 45,450 galaxies (dr11-south)
Total unique (region, SGAID) pairs indexed: 445,693
INFO:SGA.py:363:_read_catalog: Read 395,435/395,435 GROUP_PRIMARY objects from /dvs_ro/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/sample/SGA2025-beta-v1.6-dr11-south.fits
INFO:SGA.py:370:_read_catalog: Selecting 395,435/395,435 objects in region=dr

In [8]:
anchors = pd.read_csv("/pscratch/sd/q/qshimp/SGA2020-data/Anchors/VI_4000_sga152x152_complete.csv")

In [9]:
filtered_anchors, idx, sep2d = SGA_2025_Anchors(anchors, SGA_2025)
top_seps_SGA_2025, top_seps_anchors = getInfoMaxSep(anchors, SGA_2025, idx, sep2d)

Matched 3323 / 4000 anchors
Minimum: 0.00045401472350070524 arcsec
Median : 0.03432853928764917 arcsec
Maximum: 1608.427583162262 arcsec


In [10]:
top_seps_anchors

,ref_id,ra,dec,g_mag,z_mag,r_mag,Morphology,T_type,Main_type,Path,sep_arcsec
3680,1236884,179.376802,32.334058,13.758558,12.962627,13.270293,b'I',10.0,-5,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/...,9.318848
3681,1137024,124.769819,70.719790,11.887766,-1.000000,11.544065,b'I',10.0,-5,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/...,8.825619
3705,2000055,16.198340,2.120629,-1.000000,-1.000000,-1.000000,b'dIrr',10.0,-5,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/...,8.608503
3886,310441,154.679685,46.045405,15.897378,16.020426,15.534364,b'I',10.0,-5,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/...,8.120920
3143,2000085,186.922382,43.495214,-1.000000,-1.000000,-1.000000,b'dIrr',10.0,-5,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/...,7.935088
3341,596682,162.406603,11.351331,-1.000000,-1.000000,-1.000000,b'I',10.0,-5,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/...,7.478427
3879,72757,186.772611,37.143422,18.360785,-1.000000,17.881945,b'I',10.0,-5,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/...,5.990250
586,394744,181.001617,30.764404,-1.000000,-1.000000,-1.000000,b'E',-5.0,20,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/...,5.748695
1917,803810,146.373496,-0.365239,16.353888,15.171601,15.637604,b'Sb',3.0,10,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/...,5.535336
3645,445569,214.257335,43.503737,16.137250,16.550090,15.996030,b'I',10.0,-5,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/...,5.441487


In [11]:
# MATCH VIEWER
viewer_output = widgets.Output()
current_start = 0

previous_button = widgets.Button(description="← Previous", button_style="")
next_button = widgets.Button(description="Next →", button_style="")

previous_button.on_click(previous_page)
next_button.on_click(next_page)

display(widgets.HBox([previous_button, next_button]))
display(viewer_output)

update_match_view()

Output()

### 2. Filter by size

In [12]:
filtered_anchors

,SGAID,SGAGROUP,REGION,OBJNAME,PGC,SAMPLE,ELLIPSEMODE,FITMODE,BX_INIT,BY_INIT,...,R26_ERR_I,R26_ERR_Z,D26,D26_ERR,D26_REF,BA,PA,VI_REGION,Main_type,ref_id
416015,4897025,SGA2025_18222p6071,2,WISEA J120854.31+604308.2,2605896,0,0,0,123.748573,123.742592,...,0.000000,0.507585,0.561516,0.010058,r26,0.738367,144.687302,dr11-north,20,841964
410941,4870942,SGA2025_14467p5182,2,CGCG 265-029,27496,0,0,0,213.065826,212.873627,...,0.000000,0.205637,1.180915,0.011482,r26,0.761172,12.562191,dr11-north,20,742784
298836,4940926,SGA2025_18318p3145,3,WISEA J121244.06+312700.4,1947704,0,0,0,147.000000,147.000000,...,0.322619,0.291271,0.841343,0.007651,r26,0.853370,147.760666,dr11-south,20,1007845
211408,4779748,SGA2025_20034p0431,1,WISEA J132122.81+041900.3,1266564,0,0,0,163.000000,163.000000,...,0.486890,0.662428,0.944754,0.013614,r26,0.506318,54.868835,dr11-south,20,396832
210565,4777978,SGA2025_26013p2810,1,WISEA J172032.01+280603.4,3089252,0,0,0,115.000000,115.000000,...,0.701874,0.465383,0.515108,0.005797,r26,0.774242,172.224701,dr11-south,20,390387
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419739,4917666,SGA2025_17999p4956,2,MCG +08-22-051,37814,0,0,0,230.186630,227.479202,...,0.000000,0.441841,1.193630,0.011630,r26,0.614058,45.945656,dr11-north,-5,920263
307075,4955757,SGA2025_18530p0896,1,VCC 0458,39905,0,0,0,161.000000,161.000000,...,0.423668,0.405432,0.946807,0.009753,r26,0.663150,34.903229,dr11-south,-5,1064408
186748,4734287,SGA2025_18652p3185,3,WISEA J122606.04+315134.7,1968108,0,0,0,193.000000,193.000000,...,0.000000,0.427867,1.048291,0.010199,r26,0.502393,39.632694,dr11-south,-5,224150
439542,5023725,SGA2025_16346p5675,2,WISEA J105350.93+564530.3,2546877,0,0,0,115.779243,114.285843,...,0.000000,0.389143,0.550105,0.005375,r26,0.549996,35.251858,dr11-north,-5,1321251


In [13]:
# Make a set of galaxies within a certain apparent diameter range
def selectSizeBin(df, binNum, delim=None):
    '''
    1. Pick 
    '''
    cutout_size = 152
    pixel_value = 0.262 # arcseconds
    if not delim:
        delim = cutout_size * pixel_value / 60
    df = df[(df['D26'] > binNum*delim) & (df['D26'] <= (binNum+1)*delim)]
    print("galaxies less than", delim*60, "arcseconds")
    return df

In [14]:
0.663448*60

39.80688

In [15]:
remaining_anchors = selectSizeBin(filtered_anchors,0)
remaining_anchors.sort_values(by="D26")

galaxies less than 39.824 arcseconds


,SGAID,SGAGROUP,REGION,OBJNAME,PGC,SAMPLE,ELLIPSEMODE,FITMODE,BX_INIT,BY_INIT,...,R26_ERR_I,R26_ERR_Z,D26,D26_ERR,D26_REF,BA,PA,VI_REGION,Main_type,ref_id
156697,4677397,SGA2025_21812p0813,1,WISEA J143230.33+080809.9,4421246,0,0,0,119.000000,119.000000,...,0.632037,0.823257,0.421716,0.011483,z25,0.842589,168.383942,dr11-south,-5,9823
424216,4941096,SGA2025_17712p5713,2,WISEA J114829.29+570755.2,3475058,0,0,0,121.035301,120.912888,...,0.000000,0.497293,0.422368,0.013325,z25,0.565103,121.787399,dr11-north,-5,1008460
263166,4875522,SGA2025_21601p0006,1,WISEA J142403.93+000358.1,3304166,0,0,0,115.000000,115.000000,...,0.184870,0.983221,0.422751,0.005594,r26,0.853721,89.306274,dr11-south,10,759716
360619,5054224,SGA2025_16175p1296,1,LeG21,4689200,1,64,0,115.000000,115.000000,...,0.241171,0.330955,0.430603,0.006682,r26,0.764973,70.204918,dr11-south,-5,852708
282678,4910176,SGA2025_21738p0417,1,SDSS J142933.39+041013.1,4004746,0,0,0,115.000000,115.000000,...,0.256532,0.343603,0.431478,0.010292,z25,0.780443,54.416065,dr11-south,-5,892507
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198255,4755372,SGA2025_13443p2539,1,WISEA J085745.12+252346.5,1735003,0,0,0,118.000000,118.000000,...,0.197665,0.286171,0.662388,0.006000,r26,0.885715,58.765751,dr11-south,10,304383
381010,4709353,SGA2025_21098p3960,2,SDSS J140357.28+393604.2,2152660,0,0,0,136.556656,138.464432,...,0.000000,0.430652,0.662402,0.006463,r26,0.575711,127.917885,dr11-north,-5,130666
233581,4819337,SGA2025_15160p3090,3,WISEA J100625.07+305408.3,1922981,0,0,0,118.000000,118.000000,...,0.349515,0.324978,0.662763,0.006110,r26,0.978261,29.539471,dr11-south,20,548007
223354,4801510,SGA2025_02953m0858,1,WISEA J015807.42-083500.9,144449,0,0,0,120.000000,120.000000,...,0.225399,0.262064,0.663354,0.006019,r26,0.508214,75.415359,dr11-south,0,480164


### 3. Select morphology for review

In [16]:
'''
MORPHOLOGY_CODES = {
    "Elliptical": 20,
    "Lenticular": 0,
    "Spiral": 10,
    "Irregular": -5,
}

morph_display = widgets.Output()

morph_selector = widgets.Dropdown(
    options=["Elliptical", "Lenticular", "Spiral", "Irregular"],
    value="Elliptical",
    description="Morphology:"
)

def update_morph_display(change=None):
    with morph_display:
        morph_display.clear_output(wait=True)
        code = MORPHOLOGY_CODES[morph_selector.value]
        selected = remaining_anchors[remaining_anchors["Main_type"] == code].copy()

        show_cutout_grid(
            selected["SGAID"],
            cutout_index,
            ncols=N_COLS,
            figsize_per=2,
            data=selected
        )

morph_selector.observe(update_morph_display, names="value")

display(morph_selector, morph_display)

update_morph_display()
'''

'\nMORPHOLOGY_CODES = {\n    "Elliptical": 20,\n    "Lenticular": 0,\n    "Spiral": 10,\n    "Irregular": -5,\n}\n\nmorph_display = widgets.Output()\n\nmorph_selector = widgets.Dropdown(\n    options=["Elliptical", "Lenticular", "Spiral", "Irregular"],\n    value="Elliptical",\n    description="Morphology:"\n)\n\ndef update_morph_display(change=None):\n    with morph_display:\n        morph_display.clear_output(wait=True)\n        code = MORPHOLOGY_CODES[morph_selector.value]\n        selected = remaining_anchors[remaining_anchors["Main_type"] == code].copy()\n\n        show_cutout_grid(\n            selected["SGAID"],\n            cutout_index,\n            ncols=N_COLS,\n            figsize_per=2,\n            data=selected\n        )\n\nmorph_selector.observe(update_morph_display, names="value")\n\ndisplay(morph_selector, morph_display)\n\nupdate_morph_display()\n'

### 4. Separating good from bad

1. Add button clicking to each image
   a. Blue border when clicked
   b. Unhighlighted when clicked again
   c. Red border when other galaxy clicked
2. Add tag buttons 
   a. Four alternative morphology buttons
   b. Bad anchor button
3. Add note box
4. Add saving mechanism
   a. ID, RA, dec, alternative morphology, and notes should be saved
   b. Bad anchors saved separately

In [17]:
MORPHOLOGY_CODES = {
    "Elliptical": 20,
    "Lenticular": 0,
    "Spiral": 10,
    "Irregular": -5,
}

def lookup_by_morph(morph_label, df):
    code = MORPHOLOGY_CODES[morph_label]
    return df[df["Main_type"] == code].copy()

%matplotlib widget
vetter = CutoutVetter(
    morph_options=list(MORPHOLOGY_CODES.keys()),
    data_lookup_fn=lookup_by_morph,
    df=remaining_anchors,
    cutout_index=cutout_index,
    load_jpg_fn=load_jpg,
    sdss_rgb_fn=sdss_rgb,
    ncols=N_COLS,
    n_per_page=50,
    figsize_per=2,
    save_dir="/global/cfs/cdirs/desicollab/users/qshimp/anchors_small"
)

In [18]:
# Run when refining process is complete
'''
export_implicit_correct(df=remaining_anchors, morph_options=list(MORPHOLOGY_CODES.keys()),
    data_lookup_fn=lookup_by_morph, save_dir="/pscratch/sd/q/qshimp/Sorter")
'''

'\nexport_implicit_correct(df=remaining_anchors, morph_options=list(MORPHOLOGY_CODES.keys()),\n    data_lookup_fn=lookup_by_morph, save_dir="/pscratch/sd/q/qshimp/Sorter")\n'

### Next Steps

1. Add format switching (Image, model, residual)
2. Add multiple image selecting (giving group notes and morphologies)
3. Ensure saving of red border when next page is clicked

### Results

##### Initial
414 ellipticals (48 to lenticular, 6 to spiral, 2 to irregular)

493 lenticulars (59 to elliptical, 65 to spiral, 9 to irregular)

377 spirals (19 to lenticular, 9 to irregular)

520 irregulars (45 to spiral, 6 to lenticular, 2 to elliptical)

##### Change
Lost 60 lenticulars
Gained 5 ellipticals
Lost 33 irregulars
Gained 88 spirals

##### Final
487 irregulars
465 spirals
433 lenticulars
419 ellipticals

In [19]:
487+465+433+419

1804

In [20]:
def add_vi_region(catalog, sga_2025):
    """
    Add VI_REGION to `catalog` by matching each row against SGA_2025 on
    SGAID. SGAID is NOT globally unique — the same numeric ID can exist
    independently in dr11-south and dr11-north — so any SGAID with more
    than one SGA_2025 candidate is disambiguated by nearest RA/DEC rather
    than assumed to be a single unambiguous match.
    """
    from astropy.coordinates import SkyCoord
    import astropy.units as u

    sga_2025 = sga_2025.copy()
    sga_2025["SGAID"] = sga_2025["SGAID"].astype(int)
    by_sgaid = {sgaid: g for sgaid, g in sga_2025.groupby("SGAID")}

    regions = []
    unmatched, ambiguous_far = [], []

    for _, row in catalog.iterrows():
        sgaid = int(row["SGAID"])
        candidates = by_sgaid.get(sgaid)

        if candidates is None:
            regions.append(None)
            unmatched.append(sgaid)
            continue

        if len(candidates) == 1:
            regions.append(candidates.iloc[0]["VI_REGION"])
            continue

        # SGAID collision across regions -> disambiguate by nearest RA/DEC
        target = SkyCoord(ra=row["RA"] * u.deg, dec=row["DEC"] * u.deg)
        cand_coords = SkyCoord(ra=candidates["RA"].to_numpy() * u.deg,
                               dec=candidates["DEC"].to_numpy() * u.deg)
        sep = target.separation(cand_coords)
        best = sep.argmin()
        regions.append(candidates.iloc[best]["VI_REGION"])
        if sep[best] > 1 * u.arcsec:
            ambiguous_far.append((sgaid, sep[best].arcsec))

    catalog = catalog.copy()
    catalog["VI_REGION"] = regions

    if unmatched:
        u_set = sorted(set(unmatched))
        print(f"WARNING: {len(u_set)} SGAID(s) had no match in SGA_2025: {u_set[:10]}"
              f"{'...' if len(u_set) > 10 else ''}")
    if ambiguous_far:
        print(f"WARNING: {len(ambiguous_far)} SGAID collision(s) resolved but nearest "
              f"candidate was >1 arcsec away — worth spot-checking: {ambiguous_far[:5]}")

    return catalog

In [21]:
SSL_DIR = '/global/cfs/cdirs/desicollab/users/ioannis/SGA/2025/ssl'
CATALOG = '/global/cfs/cdirs/desicollab/users/qshimp/anchors/catalog_4400.csv'

cutout_index = build_cutout_index(SSL_DIR)
SGA_2025 = generate_SGA_2025(cutout_index)

anchor_catalog = pd.read_csv(CATALOG)
anchor_catalog = add_vi_region(anchor_catalog, SGA_2025)
anchor_catalog.to_csv(CATALOG, index=False)

print(anchor_catalog["VI_REGION"].value_counts(dropna=False))

  ssl-cutouts-dr11-north-chunk0000.hdf5: 41,043 galaxies (dr11-north)
  ssl-cutouts-dr11-north-chunk0001.hdf5: 41,043 galaxies (dr11-north)
  ssl-cutouts-dr11-south-chunk0000.hdf5: 45,451 galaxies (dr11-south)
  ssl-cutouts-dr11-south-chunk0001.hdf5: 45,451 galaxies (dr11-south)
  ssl-cutouts-dr11-south-chunk0002.hdf5: 45,451 galaxies (dr11-south)
  ssl-cutouts-dr11-south-chunk0003.hdf5: 45,451 galaxies (dr11-south)
  ssl-cutouts-dr11-south-chunk0004.hdf5: 45,451 galaxies (dr11-south)
  ssl-cutouts-dr11-south-chunk0005.hdf5: 45,451 galaxies (dr11-south)
  ssl-cutouts-dr11-south-chunk0006.hdf5: 45,451 galaxies (dr11-south)
  ssl-cutouts-dr11-south-chunk0007.hdf5: 45,450 galaxies (dr11-south)
Total unique (region, SGAID) pairs indexed: 445,693
INFO:SGA.py:363:_read_catalog: Read 395,435/395,435 GROUP_PRIMARY objects from /dvs_ro/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/sample/SGA2025-beta-v1.6-dr11-south.fits
INFO:SGA.py:370:_read_catalog: Selecting 395,435/395,435 objects in region=dr